<a href="https://colab.research.google.com/github/yacouba700/Bambara_AI/blob/main/MonAiBambara.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# =========================================================
# INSTALLATION
# =========================================================

!pip uninstall -y torchao
!pip install -q datasets==2.18.0 transformers accelerate peft sentencepiece

# =========================================================
# IMPORTS
# =========================================================

import shutil
import warnings
warnings.filterwarnings("ignore")

import torch
import os

from google.colab import drive, userdata
from huggingface_hub import login

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    pipeline,
    DataCollatorForLanguageModeling
)

from peft import (
    LoraConfig,
    get_peft_model,
    TaskType,
    PeftModel
)

from datasets import load_dataset

# =========================================================
# GOOGLE DRIVE
# =========================================================

drive.mount('/content/drive')

drive_model_path = "/content/drive/MyDrive/bambara-ai"
model_path = "/content/bambara-ai"

# =========================================================
# DATASET LOCAL
# =========================================================

drive_dataset = "/content/drive/MyDrive/bambara/dataset.jsonl"
local_dataset = "/content/dataset.jsonl"

# Vérifie si le fichier existe déjà en local
if not os.path.exists(local_dataset):
    shutil.copy(drive_dataset, local_dataset)
    print("Dataset copié en local")
else:
    print("Le dataset existe déjà en local, copie ignorée")

# =========================================================
# HUGGING FACE
# =========================================================

hf_token = userdata.get('HF_TOKEN')

login(token=hf_token)

print("Connexion Hugging Face réussie")

# =========================================================
# GPU
# =========================================================

print("GPU disponible :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))

# =========================================================
# MODÈLE
# =========================================================

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)

tokenizer.pad_token = tokenizer.eos_token

# =========================================================
# CHARGER MODÈLE BASE
# =========================================================

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

model.config.use_cache = False

# =========================================================
# CHARGER ANCIEN LORA SI EXISTE
# =========================================================

if os.path.exists(drive_model_path):

    print("Chargement ancien LoRA...")

    model = PeftModel.from_pretrained(
        model,
        drive_model_path,
        is_trainable=True
    )

else:

    print("Création nouveau LoRA...")

    config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=8,
        lora_alpha=16,
        lora_dropout=0.05,
        bias="none"
    )

    model = get_peft_model(model, config)

# =========================================================
# IMPORTANT
# =========================================================

model.train()

model.print_trainable_parameters()

# =========================================================
# DATASET
# =========================================================

dataset = load_dataset(
    "json",
    data_files=local_dataset,
    split="train"
)

print(dataset)

# TEST RAPIDE
dataset = dataset.shuffle(seed=42).select(range(5000))

# =========================================================
# TOKENIZATION
# =========================================================

def tokenize(example):

    tokens = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

    tokens["labels"] = tokens["input_ids"].copy()

    return tokens

tokenized_dataset = dataset.map(
    tokenize,
    remove_columns=dataset.column_names
)

# =========================================================
# DATA COLLATOR
# =========================================================

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

# =========================================================
# TRAINING ARGUMENTS
# =========================================================

training_args = TrainingArguments(
    output_dir=model_path,

    per_device_train_batch_size=4,

    gradient_accumulation_steps=1,

    num_train_epochs=2,

    learning_rate=2e-4,

    logging_steps=20,

    save_strategy="no",

    fp16=True,

    report_to="none"
)

# =========================================================
# TRAINER
# =========================================================

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)

# =========================================================
# TRAIN
# =========================================================

trainer.train()

# =========================================================
# SAVE LOCAL
# =========================================================

trainer.save_model(model_path)

tokenizer.save_pretrained(model_path)

# =========================================================
# SAVE DRIVE
# =========================================================

shutil.copytree(
    model_path,
    drive_model_path,
    dirs_exist_ok=True
)

print("Modèle sauvegardé")

# =========================================================
# TEST IA
# =========================================================

chat = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

prompt = """
### Instruction:
Traduire en bambara

### Input:
Je transmets mes savoirs à mes enfants.

### Response:
"""

result = chat(
    prompt,

    max_length=40,

    do_sample=True,

    temperature=0.7,

    top_p=0.9,

    repetition_penalty=1.3,

    eos_token_id=tokenizer.eos_token_id,

    pad_token_id=tokenizer.eos_token_id,

    return_full_text=False
)
print(result[0]["generated_text"])

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.5/510.5 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 12.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.2.0 which is incompatible.
Mounted at /content/drive
Dataset copié en local
Connexion Hugging Face réussie
GPU disponible : True
GPU : Tesla T4


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Step,Training Loss
20,1.332957
40,1.495160
60,1.511730
80,1.370095
100,1.475013
120,1.413899
140,1.450784
160,1.444652
180,1.491661
200,1.425056


Step,Training Loss
20,1.332957
40,1.495160
60,1.511730
80,1.370095
100,1.475013
120,1.413899
140,1.450784
160,1.444652
180,1.491661
200,1.425056


Passing `generation_config` together with generation-related arguments=({'do_sample', 'pad_token_id', 'top_p', 'temperature', 'repetition_penalty', 'eos_token_id', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=40) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Modèle sauvegardé
N ye n denw ma, a ka sonkunbali ye. "Ne kana ne yere senu na". Nye ni an senjigin de.!nk


In [ ]:
## -- PROJET VOCAL --##

# =========================
# 1. INSTALLATION
# =========================
!pip install -q transformers accelerate torchaudio

# =========================
# 2. IMPORTS
# =========================
from transformers import AutoProcessor, AutoModelForCTC
import torch
import torchaudio
from google.colab import files

# =========================
# 3. CHARGER MMS MODEL
# =========================
model_id = "facebook/mms-1b-all"

processor = AutoProcessor.from_pretrained(model_id)
model = AutoModelForCTC.from_pretrained(model_id)

# =========================
# 4. ACTIVER LANGUE BAMBARA
# =========================
processor.tokenizer.set_target_lang("bam")

# =========================
# 5. UPLOAD AUDIO
# =========================
print("👉 Upload ton fichier audio WAV")
uploaded = files.upload()

audio_path = list(uploaded.keys())[0]

# =========================
# 6. LECTURE AUDIO
# =========================
speech, sr = torchaudio.load(audio_path)

# Resample si nécessaire (MMS = 16kHz)
if sr != 16000:
    resampler = torchaudio.transforms.Resample(sr, 16000)
    speech = resampler(speech)

# =========================
# 7. TRANSCRIPTION
# =========================
inputs = processor(
    speech.squeeze().numpy(),
    sampling_rate=16000,
    return_tensors="pt"
)

with torch.no_grad():
    logits = model(**inputs).logits

pred_ids = torch.argmax(logits, dim=-1)
text = processor.decode(pred_ids[0])

# =========================
# 8. RESULTAT
# =========================
print("\n=======================")
print("🗣️ Transcription Bambara :")
print("=======================")
print(text)

Loading weights:   0%|          | 0/1096 [00:00<?, ?it/s]

👉 Upload ton fichier audio WAV


KeyboardInterrupt: 

In [ ]:
from google.colab import drive
import os

# Unmount if already mounted, and remove the directory if it exists and is not empty.
# This handles cases where force_remount=True might not be sufficient if the directory was pre-populated.
if os.path.exists('/content/drive') and os.path.isdir('/content/drive') and os.listdir('/content/drive'):
    # Attempt to unmount first if it's a mounted filesystem
    try:
        !fusermount -uz /content/drive
    except:
        pass # Ignore errors if it wasn't mounted or unmounting failed
    !rm -rf /content/drive

drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [3]:
import json

path = "/content/dataset.jsonl"

with open(path, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        print(json.loads(line))

        # stop après 5 lignes (optionnel)
       # if i == 30:
           # break

Streaming output truncated to the last 5000 lines.
{'text': '### Instruction:\nRéponds en bambara\n\n### Input:\npleurez plutôt sur vous-mêmes et sur vos enfants!\n\n### Response:\nAw ka kasi aw yɛrɛ ko la, ani aw denw!'}
{'text': "### Instruction:\nRéponds en bambara\n\n### Input:\nCar voici venir des jours où l'on dira: Heureuses les femmes stériles, les entrailles qui n'ont pas enfanté, et les seins qui n'ont pas nourri!\n\n### Response:\nDon dɔw natɔ filɛ, a na fɔ don minnu na, ko : Muso konanw ani minnu ma den bangin, minnu ma sin di den ma, olu kunna ka di !"}
{'text': '### Instruction:\nTraduis cette phrase en bambara\n\n### Input:\nAlors on se mettra à dire aux montagnes: Tombez sur nous! et aux collines: Couvrez-nous!\n\n### Response:\nO la, a na fɔ kulubaw ye: Aw ka bin an kan; ani kuluninw ye: Aw ka an ɲɛmadogo.'}
{'text': "### Instruction:\nDialogue entre amis en bambara\n\n### Input:\nCar si l'on traite ainsi le bois vert, qu'adviendra-t-il du sec?»\n\n### Response:\nNi ni

In [ ]:
import json
import random
import os

# =========================================================
# CHEMIN DATASET
# =========================================================

dataset_path = "/content/drive/MyDrive/bambara/dataset.jsonl"

# =========================================================
# DONNÉES DE BASE
# =========================================================

translation_data = [
    ("Bonjour", "I ni sogoma"),
    ("Comment vas-tu ?", "I ka kɛnɛ wa ?"),
    ("Je vais bien merci", "N ka kɛnɛ, i ni ce"),
    ("Où vas-tu ?", "I bɛ taa min ?"),
    ("Je suis fatigué", "N sɛkɛna"),
    ("Le professeur est arrivé", "Karamɔgɔ nana"),
    ("Je vais à l'école", "N bɛ taa kalanso la"),
    ("J'ai faim", "N kɔkɔ bɛ n la "),
    ("J'ai mal à la tête", "Kun bɛ n dimi"),
    ("Je veux manger", "N b'a fɛ ka dumuni kɛ"),
]

conversation_data = [
    # AMIS
    ("Conversation entre amis", "Salut mon frère", "Teri, i ka kɛnɛ wa ?"),
    ("Conversation entre amis", "On se voit demain ?", "an bɛ ɲɔgɔn ye sini?"),
    ("Conversation entre amis", "Je rentre à la maison", "N bɛ taa so"),

    # ÉCOLE
    ("Conversation à l'école", "Le professeur arrive", "Karamɔgɔ bɛ na"),
    ("Conversation à l'école", "Ouvre ton cahier", "I ka cahier dayɛlɛ"),
    ("Conversation à l'école", "Je n'ai pas compris", "N ma faamu"),

    # HÔPITAL
    ("Conversation à l'hôpital", "Je suis malade", "Banna bɛ n na"),
    ("Conversation à l'hôpital", "Appelez le médecin", "Dɔgɔtɔrɔ wele"),
    ("Conversation à l'hôpital", "J'ai besoin d'aide", "N manko bɛ dɛmɛ la"),

    # MARCHÉ
    ("Conversation au marché", "Combien ça coûte ?", "O ye joli ye?"),
    ("Conversation au marché", "C'est trop cher", "A ka kɛlɛ kosɛbɛ"),
]

# =========================================================
# VARIATION D'INSTRUCTION (IMPORTANT POUR ÉVITER OVERFIT)
# =========================================================

translation_instructions = [
    "Traduire en bambara",
    "Traduis cette phrase en bambara",
    "Convertis en bambara",
]

# =========================================================
# ÉCRITURE JSONL
# =========================================================

def write_sample(text):
    with open(dataset_path, "a", encoding="utf-8") as f:
        json.dump({"text": text}, f, ensure_ascii=False)
        f.write("\n")

# =========================================================
# AJOUT TRADUCTION (70%)
# =========================================================

print("Ajout des données de traduction...")

random.shuffle(translation_data)

for fr, bm in translation_data:
    instruction = random.choice(translation_instructions)

    text = f"""### Instruction:
{instruction}

### Input:
{fr}

### Response:
{bm}"""

    write_sample(text)

# =========================================================
# AJOUT CONVERSATION (30%)
# =========================================================

print("Ajout des conversations...")

random.shuffle(conversation_data)

for instr, inp, out in conversation_data:

    # légère variation pour rendre le dataset plus naturel
    variation = random.choice([
        instr,
        instr + " (dialogue)",
        instr + " entre deux personnes"
    ])

    text = f"""### Instruction:
{variation}

### Input:
{inp}

### Response:
{out}"""

    write_sample(text)

print("Ajout terminé avec succès ✔")

Ajout des données de traduction...
Ajout des conversations...
Ajout terminé avec succès ✔


In [ ]:
import torch

print("GPU disponible :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("Nom du GPU :", torch.cuda.get_device_name(0))
    print("Nombre de GPU :", torch.cuda.device_count())

    # mémoire GPU
    print("Mémoire totale :", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")

else:
    print("Aucun GPU détecté")

GPU disponible : True
Nom du GPU : Tesla T4
Nombre de GPU : 1
Mémoire totale : 14.56 GB
